# Enrich a feature class from an external service - a safe template

Three phases, and the split is the whole point:

1. **Read** a worklist with a `SearchCursor`, keyed by `OID@`, then close it.
2. **Enrich** - call the external service with **no cursor open**, so no schema lock is held
   across the network.
3. **Write back** with a fresh `UpdateCursor`, matching on `OID@`.

The same `WHERE_CLAUSE` drives phases 1 and 3, which is what makes the job **restartable**:
re-run it and it picks up only what is still outstanding. When the service is metered, that
clause is also your cost control.

---

### Read this before you run it

* **`DRY_RUN` is `True`.** Nothing is written until you change it. A dry run prints exactly
  what a live run would change, field by field.
* **`WHERE_CLAUSE` has no default and the job refuses to start without one.** Express the rows
  that still *need* work. A clause of `1=1` is rejected on purpose.
* **`MAX_ROWS` caps the run** so a mistake stays small. Hitting the cap is reported, and
  re-running continues where you left off.
* **Test against a copy first.** Phase 3 rewrites rows.

MIT licensed. Standard library plus `arcpy`.


## 1. CONFIG


In [ ]:
FEATURE_CLASS = None      # e.g. r"C:\gis\sites.gdb\locations"

# Required. The rows that still need work - and the reason this job is restartable.
WHERE_CLAUSE = None       # e.g. "CITY IS NULL OR CITY = ''"

# OID@ must be first: it is the join key between phase 1 and phase 3.
READ_FIELDS  = ["OID@", "NAME", "LAT", "LON"]
WRITE_FIELDS = ["OID@", "CITY", "ZIP"]

DRY_RUN       = True      # set False only after reading a dry-run report
MAX_ROWS      = 500       # cap per run
ONLY_IF_EMPTY = True      # never overwrite a value someone already set
LOG_PATH      = None      # e.g. r"C:\gis\enrich-log.json" - diff two runs


## 2. The engine

You should not need to edit this cell.


In [ ]:
import json
from datetime import datetime, timezone

import arcpy

OID = "OID@"


class ConfigError(Exception):
    """Raised before any data is touched, so a bad configuration costs nothing."""


# ------------------------------------------------------------------ phase 0: refuse bad input
def preflight(fc, where_clause, read_fields, write_fields, max_rows):
    """Validate before touching data. Every check here exists because the failure it prevents is
    expensive, irreversible, or both."""
    if not fc:
        raise ConfigError("FEATURE_CLASS is required.")
    if not arcpy.Exists(fc):
        raise ConfigError(f"FEATURE_CLASS does not exist: {fc}")

    if not where_clause or not str(where_clause).strip():
        raise ConfigError(
            "WHERE_CLAUSE is required and has no default. It is what makes this job restartable "
            "and what stops you paying an external service for rows that already have values. "
            "Express the rows that still NEED work, for example: \"CITY IS NULL OR CITY = ''\"")
    if str(where_clause).strip().replace(" ", "") in ("1=1", "1<>0"):
        raise ConfigError(
            "WHERE_CLAUSE selects every row. If that is genuinely intended, say so explicitly in "
            "a clause that names the fields, so the intent survives the next person reading it.")

    if OID not in read_fields:
        raise ConfigError(f"{OID} must be the first read field - it is the join key between phases.")
    if OID not in write_fields:
        raise ConfigError(f"{OID} must be present in the write fields to match rows back.")

    existing = {f.name for f in arcpy.ListFields(fc)}
    for label, fields in (("read", read_fields), ("write", write_fields)):
        missing = [f for f in fields if f != OID and f not in existing]
        if missing:
            raise ConfigError(f"{label} field(s) not on {fc}: {', '.join(missing)}")

    if not isinstance(max_rows, int) or max_rows < 1:
        raise ConfigError("MAX_ROWS must be a positive integer. It caps the blast radius of a mistake.")
    return True


# ------------------------------------------------------------------ phase 1: read
def read_worklist(fc, where_clause, read_fields, max_rows):
    """Return a list of dicts, one per row needing work. The cursor is closed before this returns -
    nothing downstream may hold it."""
    work = []
    with arcpy.da.SearchCursor(fc, read_fields, where_clause=where_clause) as cur:
        for row in cur:
            work.append(dict(zip(read_fields, row)))
            if len(work) >= max_rows:
                break
    return work


# ------------------------------------------------------------------ phase 2: the slow part
def enrich(work, lookup):
    """Call `lookup` once per row and collect the results, keyed by OID.

    `lookup` is yours to supply: it receives one row dict and returns either a dict of
    {field_name: value} to write, or None to skip the row. It runs with NO cursor open, so it may
    take as long as it needs - a network call, a rate limit, a retry - without holding a lock on
    the table.
    """
    decisions, failed = {}, []
    for row in work:
        try:
            result = lookup(row)
        except Exception as exc:                      # one bad row must not end the run
            failed.append({"oid": row[OID], "error": f"{type(exc).__name__}: {exc}"})
            continue
        if result:
            decisions[row[OID]] = result
    return decisions, failed


# ------------------------------------------------------------------ phase 3: write
def write_back(fc, where_clause, write_fields, decisions, dry_run=True, only_if_empty=True):
    """Apply decisions by OID. A fresh cursor - the phase 1 cursor is long closed, and at ArcGIS
    Pro 3.7 reusing it would raise ValueError anyway."""
    planned, changed, skipped = [], 0, 0

    with arcpy.da.UpdateCursor(fc, write_fields, where_clause=where_clause) as cur:
        for row in cur:
            oid = row[0]
            decision = decisions.get(oid)
            if not decision:
                continue

            row_changes = {}
            for i, field in enumerate(write_fields):
                if field == OID or field not in decision:
                    continue
                current, proposed = row[i], decision[field]
                if proposed is None or proposed == current:
                    continue
                # Re-check emptiness inside the write pass. WHERE_CLAUSE already restricted the set,
                # but another editor may have filled the row since phase 1 read it.
                if only_if_empty and current not in (None, "", " "):
                    skipped += 1
                    continue
                row_changes[field] = {"from": current, "to": proposed}
                row[i] = proposed

            if not row_changes:
                continue
            planned.append({"oid": oid, "changes": row_changes})
            if not dry_run:
                cur.updateRow(row)
            changed += 1

    return {"planned": planned, "changed": changed, "skipped_not_empty": skipped,
            "dry_run": bool(dry_run)}


# ------------------------------------------------------------------ reporting
def summarize(work, decisions, failed, result, max_rows):
    lines = []
    mode = "DRY RUN - nothing was written" if result["dry_run"] else "LIVE - rows were updated"
    lines.append(f"{mode}")
    lines.append(f"  rows read (cap {max_rows:,})      {len(work):,}")
    lines.append(f"  rows the lookup answered         {len(decisions):,}")
    lines.append(f"  rows the lookup failed on        {len(failed):,}")
    lines.append(f"  rows with at least one change    {result['changed']:,}")
    lines.append(f"  values skipped, already set      {result['skipped_not_empty']:,}")
    if result["planned"]:
        lines.append("")
        lines.append("  first few changes:")
        for item in result["planned"][:5]:
            for field, ch in item["changes"].items():
                lines.append(f"    OID {item['oid']:<8} {field:<22} {ch['from']!r} -> {ch['to']!r}")
    if failed:
        lines.append("")
        lines.append("  first few failures:")
        for f in failed[:5]:
            lines.append(f"    OID {f['oid']:<8} {f['error']}")
    if work and not decisions and not failed:
        lines.append("")
        lines.append("  The lookup returned nothing for every row it was given. If you have not")
        lines.append("  replaced the lookup() stub yet, that is expected - the shipped stub always")
        lines.append("  returns None so the template runs end to end without contacting anything.")
    if result["dry_run"]:
        lines.append("")
        # Only invite a live run when there is something to apply. Printing this after a run that
        # planned nothing sends you to DRY_RUN = False for a job that would do exactly nothing.
        lines.append("  Set DRY_RUN = False to apply these changes." if result["planned"]
                     else "  Nothing to apply - this dry run planned no changes.")
    if len(work) == max_rows:
        lines.append("")
        lines.append(f"  NOTE: the run stopped at the MAX_ROWS cap of {max_rows:,}. There are likely")
        lines.append("  more rows matching WHERE_CLAUSE. Re-run to continue - the clause makes it resumable.")
    return "\n".join(lines)


def write_log(path, work, decisions, failed, result):
    """A JSON record of what happened, so a re-run can be diffed against the last one."""
    payload = {
        "utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "dry_run": result["dry_run"], "rows_read": len(work),
        "rows_answered": len(decisions), "rows_changed": result["changed"],
        "skipped_not_empty": result["skipped_not_empty"],
        "planned": result["planned"], "failed": failed,
    }
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, default=str)
    return path


def run(fc, where_clause, read_fields, write_fields, lookup, max_rows=500,
        dry_run=True, only_if_empty=True, log_path=None):
    """The whole three-phase job. Returns (summary_text, result_dict)."""
    preflight(fc, where_clause, read_fields, write_fields, max_rows)
    work = read_worklist(fc, where_clause, read_fields, max_rows)
    if not work:
        return "No rows matched WHERE_CLAUSE. Nothing to do.", None
    decisions, failed = enrich(work, lookup)
    result = write_back(fc, where_clause, write_fields, decisions, dry_run, only_if_empty)
    if log_path:
        write_log(log_path, work, decisions, failed, result)
    return summarize(work, decisions, failed, result, max_rows), result


## 3. Your lookup

The only function you write.


In [ ]:
# ---------------------------------------------------------------------------------
# THE ONE FUNCTION YOU WRITE
# ---------------------------------------------------------------------------------
# Receives one row as a dict keyed by your READ_FIELDS, e.g.
#     {"OID@": 1842, "NAME": "Riverbend", "LAT": 42.41, "LON": -83.38}
# Return a dict of {field: value} to write, or None to skip the row.
#
# This runs with NO cursor open, so it may be as slow as it needs to be - a network
# call, a rate-limit pause, a retry. Raising here skips one row and the run continues.

def lookup(row):
    lat, lon = row.get("LAT"), row.get("LON")
    if lat is None or lon is None:
        return None                       # nothing to work with; skip

    # Replace this block with a real call. Example shape:
    #
    #     import urllib.request, urllib.parse, json
    #     qs = urllib.parse.urlencode({"lat": lat, "lon": lon, "token": TOKEN})
    #     with urllib.request.urlopen(f"https://example.invalid/reverse?{qs}", timeout=20) as r:
    #         payload = json.load(r)
    #     return {"CITY": payload.get("city"), "ZIP": payload.get("postal")}
    #
    # Until then the template runs end to end without contacting anything:
    return None



## 4. Run it

Safe to run as-is: `DRY_RUN` is on and the stub lookup contacts nothing.


In [ ]:
summary, result = run(
    FEATURE_CLASS, WHERE_CLAUSE, READ_FIELDS, WRITE_FIELDS, lookup,
    max_rows=MAX_ROWS, dry_run=DRY_RUN, only_if_empty=ONLY_IF_EMPTY,
    log_path=LOG_PATH,
)
print(summary)


## 5. Reading the report, and going live

A dry run tells you three numbers worth checking before you write anything:

* **rows read** - if this equals `MAX_ROWS`, there is more work outstanding.
* **rows the lookup failed on** - a handful is normal, a majority means the service or your
  credentials are wrong and you should not proceed.
* **values skipped, already set** - rows something else filled between phase 1 and phase 3.
  Non-zero here means you are not the only writer, which is worth knowing.

When the preview looks right, set `DRY_RUN = False` and re-run.

### Limits worth knowing

* `ONLY_IF_EMPTY` protects existing values. Turning it off means this template will overwrite
  data a person entered.
* The row cap is per run, not per day. A metered service will bill every call the lookup makes.
* Nothing here is transactional. If a live run is interrupted, the rows already written stay
  written - which is exactly why `WHERE_CLAUSE` matters: re-running resumes cleanly.
